In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from data_curation import DataLoader, SafeGroupKFold
from i3l_statistics import Statistics
from i3l_ml import ML
from enums import *
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from lifelines import CoxPHFitter

# Load data and separate in train, test and external set

In [3]:
dl = DataLoader()
ml = ML()
stats = Statistics()

In [4]:
modes = [
    Mode.RWD, 
    # Mode.DP, 
    # Mode.FMRAD, 
    # Mode.PYRAD, 
    # Mode.GEN
]

dataset = dl.create_dataset(
    modes=modes, 
    outcome='OS MONTHS',
    subanalysis=Subanalysis.C23
)

In [5]:
dataset

,Subject,SEX,ECOG PS,LDH,NLR,SMOKING CURRENT,SMOKING FORMER,SMOKING NEVER,PDL1 CATEGORY,PARENCHYMAL BRAIN METS AT IO START,...,BEST RESPONSE,IO LINE,IO IOCT,HISTOLOGY SQUAMOUS,ORR,DCR,CBR,PFS MONTHS,OS MONTHS,TTF MONTHS
0,GHD1030044,0,1.0,406.0,5.316808,1.0,0.0,0.0,NaN,0.0,...,2.0,1,1,0.0,1.0,1.0,1.0,20.137976,20.137976,5.059133
1,GHD1030045,1,0.0,220.0,5.151469,0.0,1.0,0.0,NaN,0.0,...,1.0,1,0,0.0,0.0,1.0,1.0,81.241787,81.241787,23.883049
2,GHD1030046,1,0.0,NaN,2.513613,1.0,0.0,0.0,2.0,0.0,...,2.0,1,0,0.0,1.0,1.0,1.0,21.254928,21.254928,9.559790
3,GHD1030047,0,0.0,237.0,4.196839,1.0,0.0,0.0,2.0,0.0,...,1.0,1,0,0.0,0.0,1.0,1.0,26.741130,26.741130,12.187911
4,GHD1030048,1,NaN,330.0,1.699781,1.0,0.0,0.0,2.0,1.0,...,0.0,1,0,0.0,0.0,0.0,0.0,1.346912,3.416557,1.379763
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2070,VHIO1020178,1,0.0,370.0,2.176471,0.0,1.0,0.0,1.0,0.0,...,3.0,2,0,0.0,1.0,1.0,1.0,92.444152,92.444152,92.444152
2071,VHIO1020179,1,0.0,NaN,NaN,0.0,1.0,0.0,NaN,0.0,...,3.0,2,0,0.0,1.0,1.0,1.0,106.504599,106.504599,106.504599
2072,VHIO1020184,0,NaN,NaN,NaN,0.0,1.0,0.0,0.0,0.0,...,2.0,2,0,0.0,1.0,1.0,1.0,6.077530,60.151117,7.588699
2073,VHIO1020198,1,1.0,NaN,NaN,0.0,1.0,0.0,NaN,0.0,...,2.0,2,0,0.0,1.0,1.0,1.0,99.310118,99.310118,23.685940


In [6]:
with open('split.json', 'r') as f:
    split = json.load(f)

train_set = dataset[dataset['Subject'].isin(split['TRAIN_SET'])].set_index('Subject')
test_set = dataset[dataset['Subject'].isin(split['TEST_SET'])].set_index('Subject')
ext_set = dataset[dataset['Subject'].str.startswith('UOC')].set_index('Subject')

In [7]:
removed = 0

while True:
    max_train_time = train_set['OS MONTHS'].max()
    max_test_time = test_set['OS MONTHS'].max()
    
    if max_test_time <= max_train_time:
        break
    removed += 1
    
    max_test_index = test_set['OS MONTHS'].idxmax()
    
    test_set = test_set.drop(max_test_index)

In [8]:
train_set.shape, test_set.shape, ext_set.shape

((1549, 24), (273, 24), (251, 24))

In [9]:
X_train, y_train = train_set.drop(columns=['OS MONTHS', 'DEATH EVENT']), train_set[['OS MONTHS', 'DEATH EVENT']]
X_test, y_test = test_set.drop(columns=['OS MONTHS', 'DEATH EVENT']), test_set[['OS MONTHS', 'DEATH EVENT']]
X_ext, y_ext = ext_set.drop(columns=['OS MONTHS', 'DEATH EVENT']), ext_set[['OS MONTHS', 'DEATH EVENT']]

In [10]:
y_train = y_train.rename(columns={'OS MONTHS': 'TIME', 'DEATH EVENT': 'EVENT'})
y_test = y_test.rename(columns={'OS MONTHS': 'TIME', 'DEATH EVENT': 'EVENT'})
y_ext = y_ext.rename(columns={'OS MONTHS': 'TIME', 'DEATH EVENT': 'EVENT'})

In [11]:
with open('submodel_features.json', 'r') as f:
    submodel_features = json.load(f)
    submodel_features = [f for f in submodel_features if f in X_train.columns]

X_train = X_train.drop(columns=submodel_features)
X_test = X_test.drop(columns=submodel_features)
X_ext = X_ext.drop(columns=submodel_features)

In [12]:
X_train_imputed, imputer = dl.impute_df(X_train)
X_test_imputed, imputer = dl.impute_df(X_test, imputer=imputer)
X_ext_imputed, imputer = dl.impute_df(X_ext, imputer=imputer)

In [13]:
X_train_imputed.columns

Index(['SEX', 'ECOG PS', 'LDH', 'NLR', 'SMOKING CURRENT', 'SMOKING FORMER',
       'SMOKING NEVER', 'PDL1 CATEGORY', 'PARENCHYMAL BRAIN METS AT IO START',
       'LIVER METS AT IO START', 'BONE METS AT IO START'],
      dtype='object')

In [14]:
X_train_scaled, scaler, to_standard_normalize, to_log_normalize = dl.normalize(X_train_imputed)
X_test_scaled, _, _, _ = dl.normalize(X_test_imputed, scaler=scaler, to_standard_normalize=to_standard_normalize, to_log_normalize=to_log_normalize)
X_ext_scaled, _, _, _ = dl.normalize(X_ext_imputed, scaler=scaler, to_standard_normalize=to_standard_normalize, to_log_normalize=to_log_normalize)

2 features log normalized
2 features standardized
2 features log normalized
2 features standardized
2 features log normalized
2 features standardized


# Train and evaluate

In [15]:
train_folds = dl.get_loco_folds(pd.Series(train_set.index))

cv = GroupKFold(n_splits=len(train_folds.unique()))

def cv_getter():
    return cv.split(X_train, y_train, groups=train_folds)

In [16]:
y_train['EVENT'] = y_train['EVENT'].astype(bool)
y_train = y_train[['EVENT', 'TIME']]

In [17]:
features = ml.coxnet_selection(
    X_train=X_train_scaled, 
    y_train=y_train.to_records(index=False), 
    cv=cv_getter, 
    folds=train_folds, 
    target_features=15, 
    uncertainty=5
)
X_train = X_train_scaled[features]
X_test = X_test_scaled[features]
X_ext = X_ext_scaled[features]

In [18]:
X_train_scaled

,SEX,ECOG PS,log_LDH,log_NLR,SMOKING CURRENT,SMOKING FORMER,SMOKING NEVER,PDL1 CATEGORY,PARENCHYMAL BRAIN METS AT IO START,LIVER METS AT IO START,BONE METS AT IO START
Subject,,,,,,,,,,,
GHD1030044,0,1,0.443497,-0.095695,1,0,0,0,0,0,1
GHD1030045,1,0,-0.344303,-0.116602,0,1,0,0,0,0,0
GHD1030046,1,0,1.249259,-0.464713,1,0,0,2,0,0,0
GHD1030047,0,0,-0.260458,-0.239344,1,0,0,2,0,0,0
GHD1030048,1,1,0.152324,-0.578079,1,0,0,2,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...
VHIO1020175,0,0,-1.277266,-0.565070,0,1,0,1,0,0,0
VHIO1020178,1,0,0.309998,-0.511314,0,1,0,1,0,0,0
VHIO1020179,1,0,0.123273,-1.912533,0,1,0,0,0,0,0


In [19]:
def coxph_cv(train_set: pd.DataFrame, cv_getter: callable, groups_train: pd.Series):
    pred_risk = np.empty(train_set.shape[0])
    pred_risk[:] = np.nan
    
    for index_train, index_test in cv_getter():
        if len(index_test) < 3:
            continue
        
        X_train_fold, X_val_fold = train_set.iloc[index_train], train_set.iloc[index_test]
        
        cph = CoxPHFitter(penalizer=0.5)
        cph.fit(X_train_fold, duration_col='TIME', event_col='EVENT')
        
        pred_risk[index_test] = cph.predict_partial_hazard(X_val_fold)
    
    return pred_risk

In [20]:
train_set = pd.concat([X_train, pd.DataFrame(y_train, index=X_train.index)], axis=1)
test_set = pd.concat([X_test, pd.DataFrame(y_test, index=X_test.index)], axis=1)
ext_set = pd.concat([X_ext, pd.DataFrame(y_ext, index=X_ext.index)], axis=1)

In [21]:
cph = CoxPHFitter(penalizer=0.5)
cph.fit(train_set, duration_col='TIME', event_col='EVENT')

<lifelines.CoxPHFitter: fitted with 1549 total observations, 417 right-censored observations>

In [22]:
risk_scores_train = cph.predict_log_partial_hazard(train_set).values
cindex_train = stats.compute_c_index_and_ci(
    time=train_set['TIME'], 
    event=train_set['EVENT'], 
    risk_score=risk_scores_train
)
risk_scores_test = cph.predict_log_partial_hazard(test_set).values
cindex_test = stats.compute_c_index_and_ci(
    time=test_set['TIME'], 
    event=test_set['EVENT'], 
    risk_score=risk_scores_test
)
risk_scores_ext = cph.predict_log_partial_hazard(ext_set).values
cindex_ext = stats.compute_c_index_and_ci(
    time=ext_set['TIME'], 
    event=ext_set['EVENT'], 
    risk_score=risk_scores_ext
)
cv_risks = coxph_cv(
    train_set=pd.concat([X_train, pd.DataFrame(y_train, index=X_train.index)], axis=1), 
    cv_getter=cv_getter, 
    groups_train=train_folds
)
cindex_cv = stats.compute_c_index_and_ci(
    time=train_set['TIME'],
    event=train_set['EVENT'],
    risk_score=cv_risks
)

In [23]:
def get_scores(values):
    return f'{values["c_index"]:.2f} ± {(values["ci"][1] - values["ci"][0]) / 2 :.2f}'

In [24]:
X_train.columns

Index(['ECOG PS', 'LIVER METS AT IO START', 'BONE METS AT IO START',
       'SMOKING CURRENT', 'PDL1 CATEGORY', 'log_NLR', 'log_LDH',
       'SMOKING FORMER', 'SEX', 'SMOKING NEVER'],
      dtype='object')

In [25]:
print(
    'Train shape', train_set.shape[0], '\n'
    'Test shape', test_set.shape[0], '\n'
    'Ext shape', ext_set.shape[0], '\n'
    'CV', get_scores(cindex_cv), '\n'
    'Train', get_scores(cindex_train), '\n',
    'Test', get_scores(cindex_test), '\n',
    'Ext', get_scores(cindex_ext), '\n'
)

Train shape 1549 
Test shape 273 
Ext shape 251 
CV 0.64 ± 0.02 
Train 0.65 ± 0.02 
 Test 0.66 ± 0.04 
 Ext 0.65 ± 0.05 

